# Train Pipeline — Blind Adversary Lab 2
## 3-Phase Curriculum RL with Topology Reward Shaping

---

### Cell 1: Cleanup & Setup

In [ ]:
import os, sys, time, random, copy, heapq, json
from pathlib import Path
from collections import deque
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

WORK_DIR = Path.cwd()
SRC_DIR = WORK_DIR.parents[1] / "src"
sys.path.insert(0, str(SRC_DIR))
sys.path.insert(0, str(WORK_DIR))

# ── Cleanup old weights ──
print("=" * 60)
print("  CLEANUP: Deleting old model weights...")
print("=" * 60)
pth_files = list(WORK_DIR.glob("*.pth"))
for p in pth_files:
    p.unlink()
    print(f"  [DELETED] {p.name}")
if not pth_files:
    print("  (No .pth files found — clean start)")

from environment import Environment, Move
from network_architect import RecurrentActorCritic, INPUT_CHANNELS, POS_DIM, HIDDEN_SIZE
from agent_loader import AgentLoader
from topology_analyzer import (
    STATIC_FULL_MAP, MOVE_ORDER, DIRS,
    _shape, _cell, _valid, _apply, _manhattan, _cell_exits, _legal,
    bfs_dist, astar, TopologyAnalyzer,
)
from state_trackers import BeliefStateTracker
from tactical_engines import IntentTracker, TrapEvaluator, DynamicModeSelector

print(f"\n  Python: {sys.version.split()[0]}")
print(f"  PyTorch: {torch.__version__}")
print(f"  Device:  {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"  Network: {INPUT_CHANNELS}-channel CNN + LSTM")
print("=" * 60)

### Cell 2: Reward Shaping & Observation Builder

In [ ]:
# ── Topology (pre-compute once) ──
STATIC_TOPO = TopologyAnalyzer()
STATIC_TOPO.analyze(STATIC_FULL_MAP)

# ── Observation builder (6-channel) ──
def _visible_cells(pos, ms, radius=5):
    H, W = ms.shape
    vis = {pos}
    r, c = pos
    for dr, dc in DIRS:
        for d in range(1, radius + 1):
            nr, nc = r + dr * d, c + dc * d
            if not (0 <= nr < H and 0 <= nc < W): break
            vis.add((nr, nc))
            if ms[nr, nc] == 1: break
    return vis

class FastBeliefTracker:
    def __init__(self, H=21, W=21):
        self.H, self.W = H, W
        self.belief = np.ones((H, W), dtype=np.float64) / (H * W)
    def update(self, my_pos, enemy_pos, ms):
        if enemy_pos is not None:
            self.belief.fill(0.0)
            er, ec = enemy_pos
            if 0 <= er < self.H and 0 <= ec < self.W:
                self.belief[er, ec] = 1.0
            return
        for r in range(self.H):
            for c in range(self.W):
                if ms[r, c] == 0: self.belief[r, c] = 0.0
        new_belief = np.zeros_like(self.belief)
        for r in range(self.H):
            for c in range(self.W):
                prob = self.belief[r, c]
                if prob <= 0: continue
                reachable = {(r, c)}
                cur = {(r, c)}
                for _ in range(2):
                    nxt_set = set()
                    for cr, cc in cur:
                        for dr, dc in DIRS:
                            nr, nc = cr + dr, cc + dc
                            if 0 <= nr < self.H and 0 <= nc < self.W:
                                nxt_set.add((nr, nc))
                    reachable |= nxt_set; cur = nxt_set
                denom = max(1, len(reachable))
                for cell in reachable:
                    new_belief[cell[0], cell[1]] += prob / denom
        self.belief = new_belief
        total = self.belief.sum()
        if total > 0: self.belief /= total

def build_obs(ms, my_pos, enemy_pos, belief, radius=5, step_num=1, max_steps=200):
    H, W = ms.shape
    vis = _visible_cells(my_pos, ms, radius)
    ch_wall = np.zeros((H, W), dtype=np.float32)
    ch_seen = np.zeros((H, W), dtype=np.float32)
    ch_fog  = np.zeros((H, W), dtype=np.float32)
    ch_enemy = np.zeros((H, W), dtype=np.float32)
    ch_belief = np.zeros((H, W), dtype=np.float32)
    ch_topo = np.zeros((H, W), dtype=np.float32)
    for r in range(H):
        for c in range(W):
            if ms[r, c] == 1: ch_wall[r, c] = 1.0
            elif (r, c) in vis: ch_seen[r, c] = 1.0
            else: ch_fog[r, c] = 1.0
    vflag = 0.0; er_n = ec_n = 0.0
    if enemy_pos is not None:
        er, ec = enemy_pos
        if 0 <= er < H and 0 <= ec < W:
            ch_enemy[er, ec] = 1.0; vflag = 1.0
            er_n = float(er) / H; ec_n = float(ec) / W
    if belief is not None:
        b_total = belief.sum()
        if b_total > 0: ch_belief = (belief / b_total).astype(np.float32)
    if STATIC_TOPO.ready:
        for r in range(H):
            for c in range(W):
                ch_topo[r, c] = STATIC_TOPO._get_topological_weight((r, c), ms)
        mx = ch_topo.max()
        if mx > 0: ch_topo /= mx
    threat_level = 1.0 - min(1.0, _manhattan(my_pos, (10, 10)) / max(H, W))
    if belief is not None and belief.sum() > 0:
        r_c = float(np.sum(np.arange(H)[:, None] * belief) / belief.sum())
        c_c = float(np.sum(np.arange(W) * belief.sum(axis=0)) / belief.sum())
        threat_level = 1.0 - min(1.0, _manhattan(my_pos, (int(r_c), int(c_c))) / max(H, W))
    game_progress = float(step_num) / float(max_steps)
    img = np.stack([ch_wall, ch_seen, ch_fog, ch_enemy, ch_belief, ch_topo], axis=0)
    pos = np.array([float(my_pos[0])/H, float(my_pos[1])/W,
                    er_n, ec_n, vflag, threat_level, game_progress], dtype=np.float32)
    return torch.from_numpy(img).unsqueeze(0), torch.from_numpy(pos).unsqueeze(0)

# ── Topology Reward Shaping ──
def pacman_reward_shaping(pacman_pos, ghost_pos, captured, prev_dist, ms):
    if captured: return 200.0
    dist = _manhattan(pacman_pos, ghost_pos)
    reward = -1.5 + (prev_dist - float(dist)) * 1.0
    if STATIC_TOPO.ready:
        if ghost_pos in STATIC_TOPO.dead_ends: reward += 10.0
        if ghost_pos in STATIC_TOPO.corridor_cells: reward += 5.0
        reward += max(0, 4 - _cell_exits(ghost_pos, ms)) * 3.0
    if dist < 4: reward += (4 - dist) * 5.0
    return reward

def ghost_reward_shaping(ghost_pos, pacman_pos, prev_dist, alive, ms):
    if not alive: return -200.0
    dist = _manhattan(ghost_pos, pacman_pos)
    reward = 1.0 - 1.0 / max(1.0, float(dist)) + (float(dist) - prev_dist) * 0.5
    if STATIC_TOPO.ready:
        if ghost_pos in STATIC_TOPO.junctions: reward += 3.0
        if ghost_pos in STATIC_TOPO.loops: reward += 5.0
        if ghost_pos in STATIC_TOPO.core and ghost_pos not in STATIC_TOPO.corridor_cells: reward += 2.0
        if ghost_pos in STATIC_TOPO.dead_ends: reward -= 10.0
        if ghost_pos in STATIC_TOPO.corridor_cells: reward -= 3.0
    if _cell_exits(ghost_pos, ms) >= 3: reward += 2.0
    return reward

# ── GAE ──
def compute_gae(rewards, values, dones, gamma, gae_lambda):
    advantages = []; gae = 0.0
    for t in reversed(range(len(rewards))):
        nv = 0.0 if t == len(rewards) - 1 else values[t + 1]
        delta = rewards[t] + gamma * nv * (1.0 - dones[t]) - values[t]
        gae = delta + gamma * gae_lambda * (1.0 - dones[t]) * gae
        advantages.insert(0, gae)
    return advantages, [a + v for a, v in zip(advantages, values)]

# ── PPO Update ──
def ppo_update(net, optimizer, obs_stack, pos_stack, old_acts, old_lp,
               advantages, returns_t, cfg, device):
    net.train(); N = len(advantages)
    indices = torch.randperm(N, device=device)
    for _ in range(cfg['update_epochs']):
        for start in range(0, N, cfg['batch_size']):
            idx = indices[start:start + cfg['batch_size']]
            b_obs = obs_stack[idx].to(device); b_pos = pos_stack[idx].to(device)
            b_act = old_acts[idx].to(device); b_adv = advantages[idx].to(device)
            b_ret = returns_t[idx].to(device); b_old_lp = old_lp[idx].to(device)
            logits, vals, _ = net(b_obs, b_pos)
            probs = F.softmax(logits, dim=-1); dist = Categorical(probs)
            new_lp = dist.log_prob(b_act).unsqueeze(-1); entropy = dist.entropy().mean()
            ratio = torch.exp(new_lp - b_old_lp)
            surr1 = ratio * b_adv
            surr2 = torch.clamp(ratio, 1.0 - cfg['clip_eps'], 1.0 + cfg['clip_eps']) * b_adv
            loss = (-torch.min(surr1, surr2).mean()
                    + cfg['vf_coef'] * F.mse_loss(vals.squeeze(-1), b_ret)
                    - cfg['entropy_coef'] * entropy)
            optimizer.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), cfg['max_grad_norm'])
            optimizer.step()

# ── Reset opponent ──
def reset_heuristic(agent):
    agent.memory_map = None
    if hasattr(agent, '_heatmap'): agent._heatmap = None
    agent._topo.__init__()
    agent.last_seen_enemy = None
    if hasattr(agent, '_enemy_direction'): agent._enemy_direction = None; agent._direction_streak = 0
    if hasattr(agent, '_steps_since_seen'): agent._steps_since_seen = 0
    agent._visit_count.clear(); agent._oscillation_moves.clear()
    agent.hidden_state = (torch.zeros(1, 1, HIDDEN_SIZE), torch.zeros(1, 1, HIDDEN_SIZE))
    if hasattr(agent, '_intent_tracker'): agent._intent_tracker = IntentTracker(max_history=8)
    if hasattr(agent, '_belief') and agent._belief is not None: agent._belief = BeliefStateTracker(21, 21)
    if hasattr(agent, '_mode_selector'): agent._mode_selector = DynamicModeSelector(hysteresis=3)
    if hasattr(agent, '_trap_evaluator'): agent._trap_evaluator.reset_cache()
    if hasattr(agent, '_history'): agent._history.clear()
    if hasattr(agent, '_enemy'): agent._enemy = None
    if hasattr(agent, '_last_enemy'): agent._last_enemy = None

GHOST_ACTION_LIST = [Move.UP, Move.DOWN, Move.LEFT, Move.RIGHT, Move.STAY]
PACMAN_ACTION_LIST = [Move.UP, Move.DOWN, Move.LEFT, Move.RIGHT,
                      Move.UP, Move.DOWN, Move.LEFT, Move.RIGHT, Move.STAY]
PACMAN_STEP_VALS = [1, 1, 1, 1, 2, 2, 2, 2, 1]

print("Cell 2 complete: reward shaping + observation builder ready.")

### Cell 3: 3-Phase Curriculum Training (4000 Episodes)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg = {
    'max_steps': 200, 'lr': 3e-4, 'entropy_coef': 0.08,
    'gamma': 0.99, 'gae_lambda': 0.95, 'clip_eps': 0.2,
    'vf_coef': 0.5, 'max_grad_norm': 0.5, 'update_epochs': 4,
    'batch_size': 64, 'obs_radius': 5, 'checkpoint_every': 500, 'seed': 42,
}
random.seed(cfg['seed']); np.random.seed(cfg['seed']); torch.manual_seed(cfg['seed'])
loader = AgentLoader(submissions_dir=str(WORK_DIR.parents[0]))

all_metrics = {
    'p1': {'ep': [], 'rew': [], 'wr': [], 'loss': [], 'act': []},
    'p2': {'ep': [], 'rew': [], 'sr': [], 'loss': [], 'act': []},
    'p3': {'ep': [], 'pr': [], 'gr': [], 'wr': [], 'sr': []},
}
total_start = time.time()

# ═══════════ PHASE 1: Train Pacman (1000 ep) ═══════════
print("\n" + "=" * 60)
print("  PHASE 1: Train Pacman vs Frozen Ghost (1000 ep)")
print("=" * 60)

net_pac = RecurrentActorCritic(action_dim=9).to(device)
bc_path = WORK_DIR / "pacman_model_bc.pth"
if bc_path.exists():
    state = torch.load(str(bc_path), map_location=device, weights_only=True)
    net_pac.load_state_dict(state, strict=False)
    print("  Warm-start: pacman_model_bc.pth loaded")

opponent_gho = loader.load_agent("24127457", "ghost")
opt_pac = optim.Adam(net_pac.parameters(), lr=cfg['lr'])
rewards_p1 = []; t1 = time.time()

pbar = tqdm(range(1, 1001), desc="Phase 1: Pacman", unit="ep",
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}] {postfix}")
for ep in pbar:
    env = Environment(max_steps=200, deterministic_starts=False,
                      capture_distance_threshold=2, pacman_speed=2)
    reset_heuristic(opponent_gho)
    _, pr, gr = env.reset()
    pac_pos = tuple(int(v) for v in pr); gho_pos = tuple(int(v) for v in gr)
    prev_dist = _manhattan(pac_pos, gho_pos)
    bo, bp, ba, bl, bv, br, bd = [], [], [], [], [], [], []
    ep_rew = 0.0; belief = FastBeliefTracker(21, 21); acts_ep = []
    for step in range(1, 201):
        po = env.get_observation("pacman", 5, 5)
        pm, pme, pen = po
        pme = tuple(int(v) for v in pme)
        pen = tuple(int(v) for v in pen) if pen is not None else None
        belief.update(pme, pen, pm)
        oi, pv = build_obs(pm, pme, pen, belief.belief, 5, step, 200)
        oi, pv = oi.to(device), pv.to(device)
        with torch.no_grad():
            lg, vl, _ = net_pac(oi, pv)
            pr_probs = F.softmax(lg, dim=-1); d = Categorical(pr_probs)
            ai = d.sample(); al = d.log_prob(ai)
        pm_move = PACMAN_ACTION_LIST[ai.item()]
        sv = min(PACMAN_STEP_VALS[ai.item()], 2)
        pac_act = (pm_move, sv) if sv > 1 else pm_move
        go = env.get_observation("ghost", 5, 5)
        gm, gme, gen = go
        gme = tuple(int(v) for v in gme)
        gen = tuple(int(v) for v in gen) if gen is not None else None
        g_move = opponent_gho.step(gm, gme, gen, step)
        g_move = loader.validate_agent_move(g_move, "ghost", "24127457")
        done, result, ns = env.step(pac_act, g_move)
        _, pr2, gr2 = ns
        pac_pos = tuple(int(v) for v in pr2); gho_pos = tuple(int(v) for v in gr2)
        cap = (result == "pacman_wins")
        cd = _manhattan(pac_pos, gho_pos)
        r = pacman_reward_shaping(pac_pos, gho_pos, cap, prev_dist, pm)
        prev_dist = cd; ep_rew += r
        bo.append(oi.squeeze(0).cpu()); bp.append(pv.squeeze(0).cpu())
        ba.append(ai.cpu()); bl.append(al.cpu())
        bv.append(vl.squeeze(-1).cpu().item()); br.append(r); bd.append(int(done))
        acts_ep.append(ai.item())
        if done: break
    if br:
        adv, ret = compute_gae(br, bv, bd, 0.99, 0.95)
        at = torch.tensor(adv, dtype=torch.float32)
        rt = torch.tensor(ret, dtype=torch.float32)
        at = (at - at.mean()) / (at.std() + 1e-8)
        ppo_update(net_pac, opt_pac, torch.stack(bo), torch.stack(bp),
                   torch.tensor(ba, dtype=torch.long).unsqueeze(-1),
                   torch.tensor(bl, dtype=torch.float32).unsqueeze(-1),
                   at, rt, cfg, device)
    rewards_p1.append(ep_rew)
    if ep % 10 == 0:
        recent = rewards_p1[-100:] if len(rewards_p1) >= 100 else rewards_p1
        cr = np.mean([1.0 if r > 100 else 0.0 for r in recent])
        avg = np.mean(recent)
        elapsed = time.time() - t1
        eta = (elapsed / ep) * (1000 - ep)
        all_metrics['p1']['ep'].append(ep)
        all_metrics['p1']['rew'].append(avg)
        all_metrics['p1']['wr'].append(cr)
        all_metrics['p1']['act'].extend(acts_ep)
        pbar.set_postfix(avgR=f"{avg:+.1f}", cap=f"{cr:.2f}", eta=f"{eta:.0f}s")

torch.save(net_pac.state_dict(), str(WORK_DIR / "pacman_model.pth"))
p1_time = time.time() - t1
print(f"  Phase 1 DONE — {p1_time:.0f}s -> pacman_model.pth")

# ═══════════ PHASE 2: Train Ghost (1000 ep) ═══════════
print("\n" + "=" * 60)
print("  PHASE 2: Train Ghost vs Frozen Pacman (1000 ep)")
print("=" * 60)

net_gho = RecurrentActorCritic(action_dim=5).to(device)
gp = WORK_DIR / "ghost_model.pth"
if gp.exists():
    state = torch.load(str(gp), map_location=device, weights_only=True)
    net_gho.load_state_dict(state, strict=False)

opponent_pac = loader.load_agent("24127457", "pacman", init_kwargs={"pacman_speed": 2})
opt_gho = optim.Adam(net_gho.parameters(), lr=cfg['lr'])
rewards_p2 = []; t2 = time.time()

pbar = tqdm(range(1, 1001), desc="Phase 2: Ghost  ", unit="ep",
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}] {postfix}")
for ep in pbar:
    env = Environment(max_steps=200, deterministic_starts=False,
                      capture_distance_threshold=2, pacman_speed=2)
    reset_heuristic(opponent_pac)
    _, pr, gr = env.reset()
    pac_pos = tuple(int(v) for v in pr); gho_pos = tuple(int(v) for v in gr)
    prev_dist = _manhattan(pac_pos, gho_pos)
    bo, bp, ba, bl, bv, br, bd = [], [], [], [], [], [], []
    ep_rew = 0.0; belief = FastBeliefTracker(21, 21); acts_ep = []
    for step in range(1, 201):
        go = env.get_observation("ghost", 5, 5)
        gm, gme, gen = go
        gme = tuple(int(v) for v in gme)
        gen = tuple(int(v) for v in gen) if gen is not None else None
        belief.update(gme, gen, gm)
        oi, pv = build_obs(gm, gme, gen, belief.belief, 5, step, 200)
        oi, pv = oi.to(device), pv.to(device)
        with torch.no_grad():
            lg, vl, _ = net_gho(oi, pv)
            pr_probs = F.softmax(lg, dim=-1); d = Categorical(pr_probs)
            ai = d.sample(); al = d.log_prob(ai)
        g_move = GHOST_ACTION_LIST[ai.item()]
        po = env.get_observation("pacman", 5, 5)
        pm, pme, pen = po
        pme = tuple(int(v) for v in pme)
        pen = tuple(int(v) for v in pen) if pen is not None else None
        pac_raw = opponent_pac.step(pm, pme, pen, step)
        pac_act = loader.validate_agent_move(pac_raw, "pacman", "24127457", 2)
        done, result, ns = env.step(pac_act, g_move)
        _, pr2, gr2 = ns
        pac_pos = tuple(int(v) for v in pr2); gho_pos = tuple(int(v) for v in gr2)
        alive = not (result == "pacman_wins")
        cd = _manhattan(pac_pos, gho_pos)
        r = ghost_reward_shaping(gho_pos, pac_pos, prev_dist, alive, gm)
        prev_dist = cd; ep_rew += r
        bo.append(oi.squeeze(0).cpu()); bp.append(pv.squeeze(0).cpu())
        ba.append(ai.cpu()); bl.append(al.cpu())
        bv.append(vl.squeeze(-1).cpu().item()); br.append(r); bd.append(int(done))
        acts_ep.append(ai.item())
        if done: break
    if br:
        adv, ret = compute_gae(br, bv, bd, 0.99, 0.95)
        at = torch.tensor(adv, dtype=torch.float32)
        rt = torch.tensor(ret, dtype=torch.float32)
        at = (at - at.mean()) / (at.std() + 1e-8)
        ppo_update(net_gho, opt_gho, torch.stack(bo), torch.stack(bp),
                   torch.tensor(ba, dtype=torch.long).unsqueeze(-1),
                   torch.tensor(bl, dtype=torch.float32).unsqueeze(-1),
                   at, rt, cfg, device)
    rewards_p2.append(ep_rew)
    if ep % 10 == 0:
        recent = rewards_p2[-100:] if len(rewards_p2) >= 100 else rewards_p2
        surv = np.mean([1.0 if r > -100 else 0.0 for r in recent])
        avg = np.mean(recent)
        elapsed = time.time() - t2
        eta = (elapsed / ep) * (1000 - ep)
        all_metrics['p2']['ep'].append(ep)
        all_metrics['p2']['rew'].append(avg)
        all_metrics['p2']['sr'].append(surv)
        all_metrics['p2']['act'].extend(acts_ep)
        pbar.set_postfix(avgR=f"{avg:+.1f}", surv=f"{surv:.2f}", eta=f"{eta:.0f}s")

torch.save(net_gho.state_dict(), str(WORK_DIR / "ghost_model.pth"))
p2_time = time.time() - t2
print(f"  Phase 2 DONE — {p2_time:.0f}s -> ghost_model.pth")

# ═══════════ PHASE 3: Joint Self-Play (2000 ep) ═══════════
print("\n" + "=" * 60)
print("  PHASE 3: Joint Self-Play (2000 ep)")
print("=" * 60)

pp = WORK_DIR / "pacman_model.pth"
gp = WORK_DIR / "ghost_model.pth"
if pp.exists():
    state = torch.load(str(pp), map_location=device, weights_only=True)
    net_pac.load_state_dict(state, strict=False)
if gp.exists():
    state = torch.load(str(gp), map_location=device, weights_only=True)
    net_gho.load_state_dict(state, strict=False)

opt_pac_j = optim.Adam(net_pac.parameters(), lr=cfg['lr'] * 0.33)
opt_gho_j = optim.Adam(net_gho.parameters(), lr=cfg['lr'] * 0.33)
pac_rew_p3, gho_rew_p3 = [], []; ent_coef = cfg['entropy_coef'] * 1.5
t3 = time.time()

pbar = tqdm(range(1, 2001), desc="Phase 3: Joint", unit="ep",
            bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}] {postfix}")
for ep in pbar:
    cur_ent = max(0.01, ent_coef * (0.9992 ** ep))
    cfg_j = copy.deepcopy(cfg); cfg_j['entropy_coef'] = cur_ent
    env = Environment(max_steps=200, deterministic_starts=False,
                      capture_distance_threshold=2, pacman_speed=2)
    _, pr, gr = env.reset()
    pac_pos = tuple(int(v) for v in pr); gho_pos = tuple(int(v) for v in gr)
    prev_dist = _manhattan(pac_pos, gho_pos)
    pbo, pbp, pba, pbl, pbv = [], [], [], [], []
    gbo, gbp, gba, gbl, gbv = [], [], [], [], []
    rw, dn = [], []; p_er, g_er = 0.0, 0.0
    bp = FastBeliefTracker(21, 21); bg = FastBeliefTracker(21, 21)
    for step in range(1, 201):
        po = env.get_observation("pacman", 5, 5)
        pm, pme, pen = po
        pme = tuple(int(v) for v in pme)
        pen = tuple(int(v) for v in pen) if pen is not None else None
        bp.update(pme, pen, pm)
        pi, ppv = build_obs(pm, pme, pen, bp.belief, 5, step, 200)
        pi, ppv = pi.to(device), ppv.to(device)
        with torch.no_grad():
            plg, pvl, _ = net_pac(pi, ppv)
            pp = F.softmax(plg, dim=-1); pd = Categorical(pp)
            pai = pd.sample(); pal = pd.log_prob(pai)
        pmv = PACMAN_ACTION_LIST[pai.item()]
        sv = min(PACMAN_STEP_VALS[pai.item()], 2)
        pac_act = (pmv, sv) if sv > 1 else pmv
        go = env.get_observation("ghost", 5, 5)
        gm, gme, gen = go
        gme = tuple(int(v) for v in gme)
        gen = tuple(int(v) for v in gen) if gen is not None else None
        bg.update(gme, gen, gm)
        gi, gpv = build_obs(gm, gme, gen, bg.belief, 5, step, 200)
        gi, gpv = gi.to(device), gpv.to(device)
        with torch.no_grad():
            glg, gvl, _ = net_gho(gi, gpv)
            gp = F.softmax(glg, dim=-1); gd = Categorical(gp)
            gai = gd.sample(); gal = gd.log_prob(gai)
        gmv = GHOST_ACTION_LIST[gai.item()]
        pac_act = loader.validate_agent_move(pac_act, "pacman", "joint", 2)
        gmv = loader.validate_agent_move(gmv, "ghost", "joint")
        done, result, ns = env.step(pac_act, gmv)
        _, pr2, gr2 = ns
        pac_pos = tuple(int(v) for v in pr2); gho_pos = tuple(int(v) for v in gr2)
        cap = (result == "pacman_wins"); alive = not cap
        cd = _manhattan(pac_pos, gho_pos)
        pr_r = pacman_reward_shaping(pac_pos, gho_pos, cap, prev_dist, pm)
        gr_r = ghost_reward_shaping(gho_pos, pac_pos, prev_dist, alive, gm)
        prev_dist = cd; p_er += pr_r; g_er += gr_r
        pbo.append(pi.squeeze(0).cpu()); pbp.append(ppv.squeeze(0).cpu())
        pba.append(pai.cpu()); pbl.append(pal.cpu())
        pbv.append(pvl.squeeze(-1).cpu().item())
        gbo.append(gi.squeeze(0).cpu()); gbp.append(gpv.squeeze(0).cpu())
        gba.append(gai.cpu()); gbl.append(gal.cpu())
        gbv.append(gvl.squeeze(-1).cpu().item())
        rw.append(pr_r); dn.append(int(done))
        if done: break
    if rw:
        ap, rp = compute_gae(rw, pbv, dn, 0.99, 0.95)
        apt = torch.tensor(ap, dtype=torch.float32)
        rpt = torch.tensor(rp, dtype=torch.float32)
        apt = (apt - apt.mean()) / (apt.std() + 1e-8)
        ppo_update(net_pac, opt_pac_j, torch.stack(pbo), torch.stack(pbp),
                   torch.tensor(pba, dtype=torch.long).unsqueeze(-1),
                   torch.tensor(pbl, dtype=torch.float32).unsqueeze(-1),
                   apt, rpt, cfg_j, device)
        gn = [-r for r in rw]
        ag, rg = compute_gae(gn, gbv, dn, 0.99, 0.95)
        agt = torch.tensor(ag, dtype=torch.float32)
        rgt = torch.tensor(rg, dtype=torch.float32)
        agt = (agt - agt.mean()) / (agt.std() + 1e-8)
        ppo_update(net_gho, opt_gho_j, torch.stack(gbo), torch.stack(gbp),
                   torch.tensor(gba, dtype=torch.long).unsqueeze(-1),
                   torch.tensor(gbl, dtype=torch.float32).unsqueeze(-1),
                   agt, rgt, cfg_j, device)
    pac_rew_p3.append(p_er); gho_rew_p3.append(g_er)
    if ep % 10 == 0:
        rp = pac_rew_p3[-100:] if len(pac_rew_p3) >= 100 else pac_rew_p3
        rg = gho_rew_p3[-100:] if len(gho_rew_p3) >= 100 else gho_rew_p3
        pc = np.mean([1.0 if r > 100 else 0.0 for r in rp])
        gs = np.mean([1.0 if r > -150 else 0.0 for r in rg])
        pa = np.mean(rp); ga = np.mean(rg)
        elapsed = time.time() - t3
        eta = (elapsed / ep) * (2000 - ep)
        all_metrics['p3']['ep'].append(ep)
        all_metrics['p3']['pr'].append(pa)
        all_metrics['p3']['gr'].append(ga)
        all_metrics['p3']['wr'].append(pc)
        all_metrics['p3']['sr'].append(gs)
        pbar.set_postfix(pacR=f"{pa:+.1f}", ghoR=f"{ga:+.1f}",
                         cap=f"{pc:.2f}", surv=f"{gs:.2f}", eta=f"{eta:.0f}s")

torch.save(net_pac.state_dict(), str(WORK_DIR / "pacman_model.pth"))
torch.save(net_gho.state_dict(), str(WORK_DIR / "ghost_model.pth"))
p3_time = time.time() - t3; total_time = time.time() - total_start
print(f"  Phase 3 DONE — {p3_time:.0f}s")
print(f"\n{'='*60}")
print(f"  ALL DONE  P1:{p1_time:.0f}s  P2:{p2_time:.0f}s  P3:{p3_time:.0f}s  TOTAL:{total_time:.0f}s")
print(f"  Models: pacman_model.pth, ghost_model.pth")
print(f"{'='*60}")

### Cell 4: Dashboard 2x2

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Training Diagnostics — 3-Phase Curriculum (4000 Episodes)",
             fontsize=16, fontweight='bold')

# ── 1: Reward Convergence ──
ax = axes[0, 0]
x1 = all_metrics['p1']['ep']; y1 = all_metrics['p1']['rew']
x2 = [e + 1000 for e in all_metrics['p2']['ep']]; y2 = all_metrics['p2']['rew']
x3 = [e + 2000 for e in all_metrics['p3']['ep']]
y3 = all_metrics['p3']['pr']; y3g = all_metrics['p3']['gr']
if y1: ax.plot(x1, y1, alpha=0.6, color='steelblue', linewidth=1.5, label='P1: Pacman')
if y2: ax.plot(x2, y2, alpha=0.6, color='coral', linewidth=1.5, label='P2: Ghost')
if y3: ax.plot(x3, y3, alpha=0.6, color='green', linewidth=1.5, label='P3: Pacman')
if y3g: ax.plot(x3, y3g, alpha=0.6, color='purple', linewidth=1.5, label='P3: Ghost')
ax.axvline(x=1000, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=2000, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Episode'); ax.set_ylabel('Avg Reward')
ax.set_title('Reward Convergence'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# ── 2: Win/Survive Rates ──
ax = axes[0, 1]
if all_metrics['p1']['wr']:
    ax.plot(x1, all_metrics['p1']['wr'], color='steelblue', linewidth=1.5, label='P1: Pacman Win')
if all_metrics['p2']['sr']:
    ax.plot(x2, all_metrics['p2']['sr'], color='coral', linewidth=1.5, label='P2: Ghost Survive')
if all_metrics['p3']['wr']:
    ax.plot(x3, all_metrics['p3']['wr'], color='green', linewidth=1.5, label='P3: Pacman Win')
if all_metrics['p3']['sr']:
    ax.plot(x3, all_metrics['p3']['sr'], color='purple', linewidth=1.5, label='P3: Ghost Survive')
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='50% equilibrium')
ax.axvline(x=1000, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=2000, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Episode'); ax.set_ylabel('Rate')
ax.set_title('Win / Survive Rates'); ax.legend(fontsize=7)
ax.set_ylim(-0.05, 1.05); ax.grid(True, alpha=0.3)

# ── 3: Loss Curves ──
ax = axes[1, 0]
if all_metrics['p1']['loss']:
    ax.plot(all_metrics['p1']['loss'], color='steelblue', alpha=0.5, linewidth=0.5, label='P1 Loss')
if all_metrics['p2']['loss']:
    off = len(all_metrics['p1']['loss'])
    ax.plot(range(off, off + len(all_metrics['p2']['loss'])),
            all_metrics['p2']['loss'], color='coral', alpha=0.5, linewidth=0.5, label='P2 Loss')
ax.set_xlabel('PPO Step'); ax.set_ylabel('Loss')
ax.set_title('Loss Smoothness'); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

# ── 4: Action Distribution ──
ax = axes[1, 1]
labels_pac = ["UPx1","DNx1","LTx1","RTx1","UPx2","DNx2","LTx2","RTx2","STAY"]
v5_mock = [18,16,17,15,12,10,11,9,2]
v6_counts = [0]*9
for a in all_metrics['p1']['act']:
    if 0 <= a < 9: v6_counts[a] += 1
v6_t = sum(v6_counts) or 1; v6_pcts = [c/v6_t*100 for c in v6_counts]
v5_t = sum(v5_mock); v5_pcts = [c/v5_t*100 for c in v5_mock]
x = np.arange(len(labels_pac)); w = 0.35
ax.bar(x - w/2, v5_pcts, w, label='V5 (est.)', color='lightcoral', alpha=0.8)
ax.bar(x + w/2, v6_pcts, w, label='V6 (current)', color='steelblue', alpha=0.8)
ax.set_xlabel('Action'); ax.set_ylabel('% Actions')
ax.set_title('Action Distribution: V5 vs V6')
ax.set_xticks(x); ax.set_xticklabels(labels_pac, rotation=45, fontsize=7)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(str(WORK_DIR / "training_dashboard.png"), dpi=150, bbox_inches='tight')
plt.show()

print(f"\nDashboard saved: {WORK_DIR / 'training_dashboard.png'}")
print(f"\n  SUMMARY:")
print(f"  Pacman final avg reward: {np.mean(rewards_p1[-100:]):+.1f}")
print(f"  Ghost  final avg reward: {np.mean(rewards_p2[-100:]):+.1f}")
print(f"  STAY action % (V6):      {v6_pcts[8]:.1f}%")
if v6_pcts[8] > 15:
    print(f"  WARNING: High STAY rate — check Safety Filter")
else:
    print(f"  OK: STAY rate within normal range")